### **Cell 1:** check the GPU (its name here is what paired_gpu must say if it's picked)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA L4, 23034 MiB


### **Cell 2:** set up (once per session)

The last line should print 2.14.1+cu130 13.0 True and the GPU's name.

In [ ]:
!command -v uv >/dev/null || pip install -q uv
!git clone -q --branch m6-2_jake https://github.com/jkelle11-source/msml612_project.git
%cd msml612_project
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name())"

/content/msml612_project
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 67 packages in 28ms
Prepared 63 packages in 24.36s
Installed 63 packages in 307ms
 + annotated-types==0.8.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + einops==0.8.2
 + filelock==4.0.12
 + fsspec==2026.9.0
 + googleapis-common-protos==1.75.5
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.3
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.24.0.43
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.30.7
 + nvidia-nvjitlink==13.4.92
 + nvidia-nvshmem-cu13=

### **Cell 3:** data (Set A from PhysioNet's open archive, then build the processed dataset)

The count should be 4000. I took that URL from PhysioNet's archive layout; it isn't written anywhere in the repo. If wget fails, upload your local data/raw/set-a.tar.gz through Colab's file panel and run !tar -xzf /content/set-a.tar.gz -C data/raw instead.

In [ ]:
!mkdir -p data/raw && wget -q -O - https://physionet.org/files/challenge-2012/1.0.0/set-a.tar.gz | tar -xz -C data/raw
!ls data/raw/set-a | wc -l
!uv run python -m data.physionet

4000


### **Cell 4:** the timing test

In [ ]:
!uv run python -m diffusion.timing

/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/2  loss 0.3986  max grad norm 7.19  [12s]
epoch 2/2  loss 0.3416  max grad norm 2.04  val MAE 0.5645  val RMSE 1.0863  [val 157s]  (forward fill 0.4366)  [12s]
base: {'epoch_seconds': 11.5, 'first_epoch_seconds': 12.4, 'validation_seconds': 157.1, 'scoring_batch_seconds': 89.91, 'first_scoring_batch_seconds': 90.11}
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/2  loss 0.3980  max grad norm 7.41  [15s]
epoch 2/2 

### **Cell 5:** G4, L4, and A100 only, the same with TF32 on

In [ ]:
!uv run python -m diffusion.timing --tf32

/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/2  loss 0.3986  max grad norm 7.20  [11s]
epoch 2/2  loss 0.3416  max grad norm 2.02  val MAE 0.5641  val RMSE 1.0832  [val 155s]  (forward fill 0.4366)  [10s]
base: {'epoch_seconds': 10.4, 'first_epoch_seconds': 11.1, 'validation_seconds': 154.8, 'scoring_batch_seconds': 72.86, 'first_scoring_batch_seconds': 72.81}
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/2  loss 0.3980  max grad norm 7.42  [14s]
epoch 2/2 

### **Cell 6:** save the records (the machine's disk is wiped when the session ends)

In [ ]:
!cat results/timing/*.json
import glob
from google.colab import files
for path in glob.glob("results/timing/*.json"):
    files.download(path)

{
 "gpu": "NVIDIA L4",
 "device": "cuda",
 "tf32": false,
 "torch": "2.14.1+cu130",
 "timed": {
  "variants": [
   "base",
   "ext1+2"
  ],
  "seed": 42,
  "epochs": 2,
  "scoring_batches": 2,
  "validation": {
   "n_samples": 5,
   "batch_size": 64
  },
  "sampler": {
   "eval_sampling_seed": 2021,
   "eval_n_samples": 100,
   "eval_batch_size": 16
  }
 },
 "timings": {
  "base": {
   "epoch_seconds": 11.5,
   "first_epoch_seconds": 12.4,
   "validation_seconds": 157.1,
   "scoring_batch_seconds": 89.91,
   "first_scoring_batch_seconds": 90.11
  },
  "ext1+2": {
   "epoch_seconds": 15.4,
   "first_epoch_seconds": 15.4,
   "validation_seconds": 159.3,
   "scoring_batch_seconds": 90.55,
   "first_scoring_batch_seconds": 90.2
  }
 },
 "estimate_hours": {
  "sweep": 136.5,
  "dry_run": 32.0
 },
 "assumes": {
  "sweep_cells": {
   "random": [
    "base",
    "ext1",
    "ext2",
    "ext1+2"
   ],
   "block-aware": [
    "base",
    "ext1"
   ]
  },
  "cost_of": {
   "base": "base",
   "ext

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>